# Cross-Domain Transfer Learning Demo

This notebook demonstrates OrcaNet's cross-domain knowledge transfer pipeline end-to-end:

1. **Setup** — configure API endpoints and import libraries
2. **Load tasks** — query OrcaMind for registered tasks
3. **Embed a new task** — create a synthetic dataset, embed with `CrossDomainEmbedder`, visualize with UMAP
4. **Retrieve similar tasks** — call `HybridRetriever` via the REST API
5. **Score transfers** — run CKA-based `FeatureTransfer` scoring and visualize per-layer heatmap
6. **LLM explanation** — get an `OrcaNetAgent` transfer recommendation
7. **Validate via OrcaLab** — submit an experiment, poll for results
8. **Results summary** — bar chart and summary table of improvement estimates

> **Prerequisites:** `docker compose -f docker-compose.dev.yml up -d` with all services healthy.

## 1. Setup

Configure service URLs from environment variables, import libraries, and verify the OrcaNet service is reachable.

In [ ]:
import json
import os

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ORCAMIND_URL = os.getenv("ORCAMIND_API_URL", "http://localhost:8000")
ORCALAB_URL = os.getenv("ORCALAB_API_URL", "http://localhost:8001")
ORCANET_URL = os.getenv("ORCANET_API_URL", "http://localhost:8002")

print(f"OrcaMind: {ORCAMIND_URL}")
print(f"OrcaLab:  {ORCALAB_URL}")
print(f"OrcaNet:  {ORCANET_URL}")

# Verify OrcaNet is reachable
try:
    with httpx.Client(timeout=5.0) as client:
        resp = client.get(f"{ORCANET_URL}/health")
        health = resp.json()
    print(f"\nOrcaNet health: {health['status']}")
    print(f"  orcamind={health.get('orcamind')}, orcalab={health.get('orcalab')}")
except Exception as e:
    print(f"\n[WARNING] OrcaNet unreachable: {e}")
    print("Start the stack with: docker compose -f docker-compose.dev.yml up -d")

## 2. Load Tasks

Query OrcaMind for available registered tasks. Each task represents a machine learning dataset with metadata such as domain, task type, number of samples, features, and classes.

In [ ]:
async def load_tasks(limit: int = 20) -> list[dict]:
    """Fetch up to *limit* registered tasks from OrcaMind."""
    async with httpx.AsyncClient(timeout=10.0) as client:
        resp = await client.get(f"{ORCAMIND_URL}/api/v1/tasks", params={"limit": limit})
        resp.raise_for_status()
        return resp.json()


try:
    tasks = await load_tasks(limit=20)
    print(f"Loaded {len(tasks)} tasks from OrcaMind.")

    if tasks:
        df_tasks = pd.DataFrame(
            [
                {
                    "task_id": t["task_id"][:8] + "…",
                    "name": t.get("name", "unknown"),
                    "domain": t.get("domain", "unknown"),
                    "task_type": t.get("task_type", "unknown"),
                    "n_samples": t.get("n_samples", 0),
                    "n_features": t.get("n_features", 0),
                    "n_classes": t.get("n_classes", 0),
                }
                for t in tasks
            ]
        )
        print()
        print(df_tasks.to_string(index=False))
    else:
        print("No tasks found — run the bootstrap script first:")
        print("  docker compose -f docker-compose.dev.yml run --rm orcamind \\")  # noqa: W605
        print("    python scripts/bootstrap_meta_dataset.py --suites cc18 --max-tasks 20")
except Exception as e:
    print(f"[ERROR] Could not load tasks: {e}")
    tasks = []

## 3. Embed a New Task

Create a synthetic dataset, compute statistical meta-features, and embed them with the `CrossDomainEmbedder`.
If at least 5 tasks are loaded, project all embeddings to 2D with UMAP to visualize domain separation.

In [ ]:
from orcanet.embeddings.cross_domain import CrossDomainEmbedder


def make_feature_vector(
    n_samples: int = 0,
    n_features: int = 0,
    n_classes: int = 0,
) -> np.ndarray:
    """Build the canonical 25-dim float32 meta-feature vector from task statistics."""
    vec = np.zeros(25, dtype=np.float32)
    vec[0] = np.log1p(float(n_samples))
    vec[1] = float(n_features)
    vec[2] = float(n_classes)
    return vec


# Embed a synthetic brain-MRI classification task
synthetic = {"n_samples": 3_000, "n_features": 64, "n_classes": 2, "domain": "medical"}
embedder = CrossDomainEmbedder()

feat_vec = make_feature_vector(**{k: v for k, v in synthetic.items() if k != "domain"})
input_tensor = torch.from_numpy(feat_vec).unsqueeze(0)  # (1, 25)
embedding = embedder.embed(input_tensor).detach().numpy().squeeze(0)  # (64,)

print(f"Synthetic task meta-features: {feat_vec[:4]}  (first 4 dims)")
print(f"Embedding shape : {embedding.shape}")
print(f"Embedding L2 norm: {np.linalg.norm(embedding):.4f}  (should be ≈1.0)")

# --- UMAP visualization (requires umap-learn) ---
if len(tasks) >= 5:
    try:
        from umap import UMAP  # type: ignore[import]

        vecs = np.array(
            [make_feature_vector(t.get("n_samples", 0), t.get("n_features", 0), t.get("n_classes", 0))
             for t in tasks]
        )
        domains = [t.get("domain", "unknown") for t in tasks]

        all_tensors = torch.from_numpy(vecs)  # (N, 25)
        all_embeddings = embedder.embed(all_tensors).detach().numpy()  # (N, 64)

        # Append the synthetic task embedding
        all_embeddings = np.vstack([all_embeddings, embedding])
        domains.append(synthetic["domain"] + " (new)")

        reducer = UMAP(n_components=2, random_state=42)
        reduced = reducer.fit_transform(all_embeddings)

        unique_domains = sorted(set(domains))
        color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(unique_domains), 1)))
        color_map = dict(zip(unique_domains, color_cycle))

        plt.figure(figsize=(9, 6))
        for domain in unique_domains:
            idxs = [i for i, d in enumerate(domains) if d == domain]
            marker = "*" if "new" in domain else "o"
            size = 180 if "new" in domain else 60
            plt.scatter(
                reduced[idxs, 0], reduced[idxs, 1],
                label=domain, alpha=0.8, s=size, marker=marker,
                color=color_map[domain],
            )
        plt.title("Cross-Domain Task Embeddings (UMAP 2-D projection)")
        plt.xlabel("UMAP-1")
        plt.ylabel("UMAP-2")
        plt.legend(title="Domain", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("umap-learn not installed — skipping 2D visualization.")
        print("  pip install umap-learn")
else:
    print(f"Only {len(tasks)} tasks loaded — need ≥ 5 for UMAP visualization.")

## 4. Retrieve Similar Tasks

Call `POST /api/v1/retrieve` on OrcaNet and display the top-5 most similar tasks by cosine similarity in the shared embedding space.

In [ ]:
async def retrieve_similar_tasks(
    task_id: str,
    description: str | None = None,
    top_k: int = 5,
) -> list[dict]:
    """Return the top-k similar tasks for the given task_id via OrcaNet."""
    payload: dict = {"task_id": task_id, "top_k": top_k}
    if description:
        payload["query_description"] = description
    async with httpx.AsyncClient(timeout=30.0) as client:
        resp = await client.post(f"{ORCANET_URL}/api/v1/retrieve", json=payload)
        resp.raise_for_status()
        return resp.json()


similar_tasks: list[dict] = []

if tasks:
    query_task = tasks[0]
    query_id = query_task["task_id"]
    query_name = query_task.get("name", query_id[:8])
    print(f"Retrieving similar tasks for: '{query_name}'")
    print()

    try:
        similar_tasks = await retrieve_similar_tasks(
            query_id,
            description=f"{query_task.get('domain', '')} {query_task.get('task_type', '')}".strip(),
            top_k=5,
        )

        if similar_tasks:
            df_similar = pd.DataFrame(
                [
                    {
                        "rank": r.get("rank", i + 1),
                        "task_id": str(r["task_id"])[:8] + "…",
                        "similarity": f"{r['score']:.4f}",
                    }
                    for i, r in enumerate(similar_tasks)
                ]
            )
            print(df_similar.to_string(index=False))
        else:
            print("No similar tasks returned — FAISS index may be empty.")
            print("Run bootstrap_meta_dataset.py to populate the index.")
    except httpx.HTTPStatusError as e:
        status = e.response.status_code
        print(f"[{status}] {e.response.text[:200]}")
        if status == 503:
            print("Retriever not initialised — FAISS index is empty.")
    except (RuntimeError, Exception) as e:
        print(f"[ERROR] Could not retrieve similar tasks: {e}")
else:
    print("No tasks loaded. Run Cell 2 first.")

## 5. Score Transfers

For each of the top-3 retrieved candidate tasks, call `POST /api/v1/transfer/score` with the `feature` strategy to obtain CKA-based transfer scores. Visualize overall scores and a per-layer CKA heatmap for the best candidate.

In [ ]:
async def score_transfer(
    source_task_id: str,
    target_task_id: str,
    strategy: str = "feature",
) -> dict:
    """Score transferability from source to target via OrcaNet."""
    payload = {
        "source_task_id": source_task_id,
        "target_task_id": target_task_id,
        "strategy": strategy,
    }
    async with httpx.AsyncClient(timeout=30.0) as client:
        resp = await client.post(f"{ORCANET_URL}/api/v1/transfer/score", json=payload)
        resp.raise_for_status()
        return resp.json()


transfer_scores: list[dict] = []

if len(tasks) >= 4:
    target_task = tasks[0]
    target_id = target_task["task_id"]
    target_name = target_task.get("name", target_id[:8])

    # Use retrieved candidates if available, else fall back to the next 3 tasks
    candidate_ids = (
        [str(r["task_id"]) for r in similar_tasks[:3]]
        if similar_tasks
        else [t["task_id"] for t in tasks[1:4]]
    )

    print(f"Scoring transfers → '{target_name}'")
    print()

    for src_id in candidate_ids:
        try:
            score = await score_transfer(src_id, target_id)
            src_name = next(
                (t.get("name", src_id[:8]) for t in tasks if t["task_id"] == src_id),
                src_id[:8],
            )
            transfer_scores.append(
                {
                    "source_id": src_id,
                    "source_name": src_name,
                    "overall": score["overall"],
                    "layer_scores": score.get("layer_scores", {}),
                    "reasoning": score.get("reasoning", ""),
                }
            )
            print(f"  {src_name:<30s}  overall={score['overall']:.3f}  {score.get('reasoning', '')}")
        except httpx.HTTPStatusError as e:
            print(f"  [SKIP] {src_id[:8]}… — {e.response.status_code}: {e.response.text[:80]}")
        except (RuntimeError, Exception) as e:
            print(f"  [SKIP] {src_id[:8]}… — {e}")

    if transfer_scores:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: overall scores bar chart
        names = [s["source_name"] for s in transfer_scores]
        overall_vals = [s["overall"] for s in transfer_scores]
        colors = ["#2196F3", "#4CAF50", "#FF9800"][: len(transfer_scores)]
        axes[0].barh(names, overall_vals, color=colors)
        axes[0].set_xlim(0, 1.0)
        axes[0].set_xlabel("Transfer Score (CKA)")
        axes[0].set_title(f"Overall Transfer Scores → {target_name}")
        axes[0].axvline(0.5, linestyle="--", color="red", alpha=0.5, label="threshold=0.5")
        axes[0].legend()

        # Right: per-layer CKA heatmap for the best candidate
        best = max(transfer_scores, key=lambda x: x["overall"])
        layer_scores = best["layer_scores"]
        if layer_scores:
            layers = list(layer_scores.keys())
            vals = np.array([[layer_scores[layer] for layer in layers]])
            try:
                import seaborn as sns  # type: ignore[import]
                sns.heatmap(
                    vals,
                    annot=True,
                    fmt=".2f",
                    xticklabels=layers,
                    yticklabels=[best["source_name"]],
                    ax=axes[1],
                    vmin=0,
                    vmax=1,
                    cmap="YlOrRd",
                )
            except ImportError:
                axes[1].text(
                    0.5, 0.5, "seaborn not installed — pip install seaborn",
                    ha="center", va="center", transform=axes[1].transAxes,
                )
            axes[1].set_title(f"Per-Layer CKA: {best['source_name']} → {target_name}")
        else:
            axes[1].text(
                0.5, 0.5, "No per-layer scores available",
                ha="center", va="center", transform=axes[1].transAxes,
            )
            axes[1].set_title("Per-Layer CKA")

        plt.tight_layout()
        plt.show()
        print(f"\nBest candidate: {best['source_name']} (score={best['overall']:.3f})")
else:
    print("Need ≥ 4 tasks loaded. Run Cell 2 first.")

## 6. LLM Explanation

Call `POST /api/v1/transfer/recommend` to let the `OrcaNetAgent` reason about the best transfer candidates and return a structured recommendation with an explanation.

> **Requires** `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` environment variable.

In [ ]:
async def recommend_transfer(
    target_task_id: str,
    description: str,
    top_k: int = 3,
) -> dict:
    """Ask the OrcaNet reasoning agent for the best transfer sources."""
    payload = {
        "target_task_id": target_task_id,
        "query_description": description,
        "top_k": top_k,
    }
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(f"{ORCANET_URL}/api/v1/transfer/recommend", json=payload)
        resp.raise_for_status()
        return resp.json()


llm_recommendation: dict = {}

if tasks:
    target_task = tasks[0]
    target_id = target_task["task_id"]
    description = (
        f"{target_task.get('domain', 'vision')} "
        f"{target_task.get('task_type', 'classification')} "
        f"with {target_task.get('n_classes', '?')} classes"
    )
    print(f"Querying OrcaNet agent for: '{description}'")
    print()

    try:
        llm_recommendation = await recommend_transfer(target_id, description, top_k=3)

        strategy = llm_recommendation.get("recommended_strategy", "N/A")
        improvement = llm_recommendation.get("expected_improvement", 0)
        confidence = llm_recommendation.get("confidence", 0)
        explanation = llm_recommendation.get("explanation", "N/A")
        sources = llm_recommendation.get("top_sources", [])

        print("═" * 62)
        print("  OrcaNet Transfer Recommendation")
        print("═" * 62)
        print(f"  Strategy:             {strategy}")
        print(f"  Expected Improvement: {improvement:.1%}")
        print(f"  Confidence:           {confidence:.1%}")
        print()
        print("  Explanation:")
        for line in explanation.split(". "):
            if line:
                print(f"    {line.strip()}.")
        print()
        print("  Top Source Tasks:")
        for i, src in enumerate(sources, 1):
            name = src.get("task_name", src.get("task_id", "unknown")[:8])
            sim = src.get("similarity_score", 0)
            tscore = src.get("transfer_score", 0)
            reasoning = src.get("reasoning", "")
            print(f"  {i}. {name}")
            print(f"     similarity={sim:.3f}  transfer_score={tscore:.3f}")
            if reasoning:
                print(f"     {reasoning}")
        print("═" * 62)

    except httpx.HTTPStatusError as e:
        if e.response.status_code == 502:
            print("[502] LLM agent error — check that OPENAI_API_KEY is set.")
        else:
            print(f"[{e.response.status_code}] {e.response.text[:200]}")
    except (RuntimeError, Exception) as e:
        print(f"[ERROR] Recommendation failed: {e}")
else:
    print("No tasks loaded. Run Cell 2 first.")

## 7. Validate via OrcaLab

Submit the best candidate transfer to `POST /api/v1/transfer/validate`. OrcaNet will score the transfer, then dispatch an experiment to OrcaLab to empirically measure the improvement over a baseline trained from scratch.

In [ ]:
async def validate_transfer(
    source_task_id: str,
    target_task_id: str,
    strategy: str = "feature",
    run_validation: bool = True,
) -> dict:
    """Run the full three-way pipeline: score + OrcaLab experiment + persist mapping."""
    payload = {
        "source_task_id": source_task_id,
        "target_task_id": target_task_id,
        "strategy": strategy,
        "run_validation": run_validation,
    }
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(f"{ORCANET_URL}/api/v1/transfer/validate", json=payload)
        resp.raise_for_status()
        return resp.json()


validation_result: dict = {}

if len(tasks) >= 2:
    target_task = tasks[0]
    target_id = target_task["task_id"]
    target_name = target_task.get("name", target_id[:8])

    # Use best transfer candidate if available, otherwise the second task
    if transfer_scores:
        best_source = max(transfer_scores, key=lambda x: x["overall"])
        source_id = best_source["source_id"]
        source_name = best_source["source_name"]
    else:
        source_id = tasks[1]["task_id"]
        source_name = tasks[1].get("name", source_id[:8])

    print(f"Validating: '{source_name}' → '{target_name}'")
    print("Submitting experiment to OrcaLab… (may take several minutes)")
    print()

    try:
        validation_result = await validate_transfer(source_id, target_id)

        score_info = validation_result.get("score", {})
        exp_result = validation_result.get("experiment_result")
        improvement = validation_result.get("improvement_over_baseline")

        print(f"Transfer score:        {score_info.get('overall', 0):.3f}")
        print(f"Recommended layers:    {score_info.get('recommended_layers', [])}")

        if exp_result:
            metrics = exp_result.get("metrics", {})
            exp_id = exp_result.get("experiment_id", "N/A")
            accuracy = metrics.get("accuracy")
            baseline = metrics.get("baseline_accuracy")
            print(f"Experiment ID:         {exp_id}")
            if accuracy is not None:
                print(f"Validated accuracy:    {accuracy:.4f}")
            if baseline is not None:
                print(f"Baseline accuracy:     {baseline:.4f}")
            if improvement is not None:
                sign = "+" if improvement >= 0 else ""
                print(f"Improvement:           {sign}{improvement:.4f} ({sign}{improvement * 100:.2f}%)")
        else:
            print("Validation experiment not run (score below threshold or OrcaLab unavailable).")

    except httpx.HTTPStatusError as e:
        print(f"[{e.response.status_code}] Validation failed: {e.response.text[:200]}")
    except (RuntimeError, Exception) as e:
        print(f"[ERROR] Validation failed: {e}")
else:
    print("Need ≥ 2 tasks loaded. Run Cell 2 first.")

## 8. Results Summary

Produce a bar chart of the estimated improvement over baseline for each candidate and print the full summary table: source task → target task → strategy → transfer score → improvement.

In [ ]:
if transfer_scores and tasks:
    target_task = tasks[0]
    target_name = target_task.get("name", target_task["task_id"][:8])

    # Build summary rows — use validated improvement if available, else estimate from score
    summary_rows = []
    for score_info in transfer_scores:
        validated_improvement: float | None = None
        if (
            validation_result
            and validation_result.get("improvement_over_baseline") is not None
            and score_info["source_id"] == (
                max(transfer_scores, key=lambda x: x["overall"])["source_id"]
            )
        ):
            validated_improvement = validation_result["improvement_over_baseline"]

        improvement = (
            validated_improvement
            if validated_improvement is not None
            else score_info["overall"] * 0.15  # rough estimate: 15% of transfer score
        )
        label = "validated" if validated_improvement is not None else "estimated"

        summary_rows.append(
            {
                "Source Task": score_info["source_name"],
                "Target Task": target_name,
                "Strategy": "feature",
                "Transfer Score": f"{score_info['overall']:.3f}",
                "Improvement": f"{improvement:+.3f}",
                "Type": label,
            }
        )

    df_summary = pd.DataFrame(summary_rows)
    print("Transfer Learning Results Summary")
    print("=" * 70)
    print(df_summary.to_string(index=False))

    # Bar chart
    improvements = [
        float(row["Improvement"].replace("+", "")) for row in summary_rows
    ]
    source_names = [row["Source Task"] for row in summary_rows]
    bar_colors = ["#2196F3", "#4CAF50", "#FF9800"][: len(summary_rows)]
    labels = [row["Type"] for row in summary_rows]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(source_names, improvements, color=bar_colors)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_ylabel("Improvement over Baseline")
    ax.set_title(f"Transfer Improvement → {target_name}")
    ax.set_ylim(min(improvements) - 0.02, max(improvements) + 0.04)

    for bar, val, lbl in zip(bars, improvements, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{val:+.3f}\n({lbl})",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    plt.tight_layout()
    plt.show()

else:
    # Demo output when the stack is not running
    print("Run cells 2–7 with a live Orca stack to see real results.")
    print()
    demo = pd.DataFrame(
        {
            "Source Task": ["ImageNet-Classify", "CIFAR-10", "MedMNIST"],
            "Target Task": ["brain-mri-seg"] * 3,
            "Strategy": ["feature", "weight", "architecture"],
            "Transfer Score": [0.82, 0.71, 0.64],
            "Improvement": ["+0.123", "+0.089", "+0.061"],
            "Type": ["validated", "estimated", "estimated"],
        }
    )
    print("[Demo output]")
    print(demo.to_string(index=False))